# Stage 1 — Text extraction with PaddleOCR

**Goal.** Inspect the cached PaddleOCR output (text, confidence and bounding box per line), the reading-order reconstruction (including two-column layouts), OCR time per CV, mean confidence and failed images.

Logic: `src/ocr_extraction.py`. The full OCR run is executed as a script (`python -m src.ocr_extraction`), not in this notebook.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import utils

utils.set_seed()
print("SEED:", utils.SEED, "| device:", utils.get_device(), "| root:", utils.ROOT)

## OCR setup

`src/ocr_extraction.py` runs PaddleOCR 3.x with the assignment settings: `use_doc_orientation_classify=False`, `use_doc_unwarping=False`, `use_textline_orientation=False`, on **CPU** (`device="cpu"`).

The detection and recognition models are named explicitly. In PaddleOCR 3.7, `lang="en"` alone resolves to PP-OCRv6_medium, and `ocr_version="PP-OCRv5"` mixes the server detector with the mobile recogniser. The two PP-OCRv5 configurations compared below are:

| Config | Detection | Recognition |
|---|---|---|
| mobile | `PP-OCRv5_mobile_det` | `en_PP-OCRv5_mobile_rec` (English model) |
| server | `PP-OCRv5_server_det` | `PP-OCRv5_server_rec` |

Each CV produces `data/ocr/<id>.json` with every detected line (text, confidence score, bounding box `[x1, y1, x2, y2]`) and the processing time. The full run is a script, not a notebook cell:

```bash
caffeinate -i python -m src.ocr_extraction          # resumable: existing JSONs are skipped
python -m src.ocr_extraction --benchmark            # mobile vs server on 20 CVs + parallel test
python -m src.ocr_extraction --figures              # reading-order figures
```

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from src.ocr_extraction import BENCHMARK_DIR, load_ocr, ocr_text, reading_order
from src.visualization import plot_reading_order

summary_path = utils.METRICS_DIR / "ocr_summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    display(pd.Series({k: v for k, v in summary.items() if k not in ("hardware", "failed_ids")}, name="value").to_frame())
    display(pd.Series(summary["hardware"], name="hardware").to_frame())
else:
    print("Full OCR run not done yet: run `python -m src.ocr_extraction`")

## Reading order

PaddleOCR returns text lines in detection order, which is not guaranteed to be the reading order. `reading_order()` rebuilds it from the boxes:

1. **Column detection.** Scan candidate vertical split lines in the middle 20–80% of the page width and keep the one crossed by the fewest boxes. It is a two-column layout only if few boxes cross it, both sides hold at least 4 boxes, and the two sides overlap vertically (they sit side by side, not one above the other).
2. **Bands.** Boxes that cross the split (full-width headers or section titles) cut the page into horizontal bands.
3. **Within each band**, read the **left column completely, then the right column**.
4. **Within a column**, group boxes into visual lines (vertical overlap ≥ 50% of the smaller box height), top to bottom, each line left to right, and join them with a space. A job title and its dates on the same visual line become one text line.

The reading order is computed when the JSON is loaded, so changing it does not require re-running the OCR. Below: one CV with its boxes numbered in reading order (one number per visual line), and the extracted text.

In [ ]:
cv_dir = utils.OCR_DIR if any(utils.OCR_DIR.glob("*.json")) else BENCHMARK_DIR / "mobile"
cv_id = sorted(p.stem for p in cv_dir.glob("*.json"))[0]
record = load_ocr(cv_id, cv_dir)
order = reading_order(record)

fig, ax = plt.subplots(figsize=(8, 8))
plot_reading_order(record, order, Image.open(utils.RAW_IMAGES_DIR / f"{cv_id}.png"), ax,
                   title=f"id {cv_id} · layout: {order['layout']} · {record['seconds']:.1f}s · mean confidence {record['mean_score']:.3f}")
plt.show()
print(ocr_text(record))

### Three occupations

Saved by `python -m src.ocr_extraction --figures` in `output/figures/ocr_reading_order_<occupation>.png`. Check that each job title is on the same numbered line as its dates.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 8))
for ax, occ in zip(axes, ["registered_nurses", "software_developers", "construction_laborers"]):
    ax.imshow(Image.open(utils.FIGURES_DIR / f"ocr_reading_order_{occ}.png"))
    ax.axis("off")
    ax.grid(False)
plt.tight_layout()
plt.show()

### Two-column check

Every PopResume CV seen so far uses the same single-column template, so the column detector should almost never fire on real data (the full run reports how many layouts it found). To check that it works when needed, a synthetic page puts the bodies of two real CVs side by side under a full-width banner. The banner (blue) is read first, then the whole left column (orange), then the whole right column (green).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
ax.imshow(Image.open(utils.FIGURES_DIR / "ocr_reading_order_two_column_check.png"))
ax.axis("off")
ax.grid(False)
plt.show()

## PP-OCRv5 mobile vs server

Same 20 CVs (4 per occupation, seed 42), one process with 8 CPU threads. CER and WER compare the OCR text, in reading order, with the reference text after the same normalisation (`src/ocr_quality.normalize_text`: lowercase, unified dashes and quotes, no bullet markers, single spaces around `|`, collapsed whitespace). `pct_years_correct` is the share of four-digit years in the reference text that also appear in the OCR text, which matters most here because years are the evidence for experience. The `*_after_date_fix` columns apply `preprocessing.repair_date_ranges` first.

In [ ]:
comparison = pd.read_csv(utils.TABLES_DIR / "ocr_model_comparison.csv")
comparison

In [ ]:
per_cv = pd.read_csv(utils.TABLES_DIR / "ocr_model_comparison_per_cv.csv", dtype={"id": str})
per_cv.pivot(index="id", columns="model", values=["seconds", "cer", "years_ok"]).round(3)

**Findings (20 CVs, Apple M2, 8 GB RAM):**

- **Speed.** Mobile takes about 5.8 s per CV in one process; server takes about 187 s and peaks at about 5 GB of RAM. Server would need about 8–9 days for 4,000 CVs on this machine.
- **Raw text quality.** Server is slightly better as is (CER 0.05% vs 0.19%, WER 0.6% vs 1.7%) because mobile drops the en dash in about half of the date ranges (`2022.10–2023.01` → `2022.102023.01`). Only 81.6% of reference years then appear as separate years.
- **After the date-range repair.** The digits are read correctly, so `preprocessing.repair_date_ranges` re-inserts the dash. Mobile then has **CER 0.00% and 100% of years**, slightly better than server (CER 0.05%).

**Decision: PP-OCRv5 mobile** (`PP-OCRv5_mobile_det` + `en_PP-OCRv5_mobile_rec`), with the date-range repair applied in preprocessing. It is 32× faster, and its only systematic error is fixed with one regex.

## Parallel processes

PaddlePaddle already runs multi-threaded on CPU, but one process does not use the 8 cores well: 1×8 and 1×4 threads are equally fast. Splitting the cores across processes helps: **4 processes × 2 threads are 2.15× faster** than 1 × 8 (2.6 s per CV of wall time on the same 20 CVs, including model loading). Four mobile processes fit in 8 GB of RAM. This is the default of `python -m src.ocr_extraction`; use `--workers 2 --threads 4` if the machine starts swapping.

In [ ]:
pd.read_csv(utils.TABLES_DIR / "ocr_parallel_test.csv")